# DEARLIBS Battery Model Implementation in Python

## Initialization

In [ ]:
# Imports

import numpy as np
import sympy as sp
from sympy import symbols, diff, simplify, Function, sqrt, sinh, tanh, exp, sympify, Eq, IndexedBase
from sympy.utilities.lambdify import lambdify
from scipy.integrate import solve_ivp
from pyswarms.single import GlobalBestPSO
import matplotlib.pyplot as plt


### Model parameters, Applied current, and Number of node points.

In [2]:
# Symbolic variables and parameters

t = sp.symbols('t')

F, R, T, c0, ctp, ctn = symbols('F R T c0 ctp ctn')

# Number of parameters to be identified (Input the number of parameters to be identified)
n_vars = 8
kk = sp.symbols('k1:%d' % (n_vars+1))  # k1, k2, ..., k8
# kk = sp.symbols('k1:9')  # k1 to k8

# C-rate (Put your C-rate)
Crate = -1

# Experimental data (Users can input their experimental conditions
Numexp = 63
Totexp = 3100

# Node number (Change # of node- N: Cathode, M: Membrance, NM: Cathode) (Put your number of node points)
N, M, NM = 2, 2, 2

# Design parameters
ep, es, en = 0.335, 0.47, 0.25
brugp, brugs, brugn = 2.43, 2.57, 2.91
lp, ls, ln1 = 75.6e-6, 12e-6, 85.2e-6
Rpp, Rpn = 5.22e-6, 5.86e-6
F = 96487
R_const = 8.3143
t1 = 0.363
ap = (3/Rpp)*(1-ep)
an = (3/Rpn)*(1-en)
T = 298.15
Acell = 0.11
Capa = 5
iapp = Capa * Crate / Acell

# Transport params symbolic with kk
c0 = 1000
D1 = kk[0] * 1e-9
Kappa = kk[1]
ctp = 51765
ctn = 29583
Dbulk = D1
sigmap = kk[2]
sigman = kk[3]
Dsp = kk[4] * 1e-15
Dsn = kk[5] * 1e-14

Keffp = Kappa * (ep ** brugp)
Keffs = Kappa * (es ** brugs)
Keffn = Kappa * (en ** brugn)
D2pos = (ep ** brugp) * Dbulk
D2sep = (es ** brugs) * Dbulk
D2neg = (en ** brugn) * Dbulk

kp = kk[6] * 1e-11
kn = kk[7] * 1e-12

h = lp/(N+1)
h2 = ls/(M+1)
h3 = ln1/(NM+1)

# Symbolic state variables X_i(t)
Nt = (
    1 + N + 1 + M + 1 + NM + 1 +   # u1
    N + NM +                      # u2
    N + NM +                      # u3
    N + 2 + NM + 2 +              # u4
    1 + N + 1 + M + 1 + NM + 1    # u5
)

X = [Function(f'X{i+1}')(t) for i in range(Nt)]


## Define Equations

In [3]:
# Variable lengths
len_u1 = 1 + N + 1 + M + 1 + NM + 1     # Electrolyte concentration
len_u2 = N + NM                        # Surface concentration
len_u3 = N + NM                        # Average concentration
len_u4 = N + 2 + NM + 2                # Solid-phase potential
len_u5 = len_u1                        # Liquid potential

offset_u1 = 0
offset_u2 = offset_u1 + len_u1
offset_u3 = offset_u2 + len_u2
offset_u4 = offset_u3 + len_u3
offset_u5 = offset_u4 + len_u4

assert offset_u5 + len_u5 == Nt, "Length accounting error in X construction!"

# Slice variables
u1 = X[offset_u1 : offset_u1 + len_u1]
u2 = X[offset_u2 : offset_u2 + len_u2]
u3 = X[offset_u3 : offset_u3 + len_u3]
u4 = X[offset_u4 : offset_u4 + len_u4]
u5 = X[offset_u5 : offset_u5 + len_u5]

# positive region node count incl. both boundaries
npos = N + 2           # 1 + N + 1
nsep = M + 1           # interior + right boundary
nneg = NM + 1          # interior + right boundary

# u1 segmentation
u1_pos = u1[0 : npos]                         # length N+2
u1_sep = u1[npos : npos + nsep]               # length M+1
u1_neg = u1[npos + nsep : npos + nsep + nneg] # length NM+1

# u5 segmentation (same structure)
u5_pos = u5[0 : npos]
u5_sep = u5[npos : npos + nsep]
u5_neg = u5[npos + nsep : npos + nsep + nneg]

# u2 segmentation: [N | NM]
u2_pos = u2[0 : N]
u2_neg = u2[N : N + NM]

# u3 segmentation: [N | NM]
u3_pos = u3[0 : N]
u3_neg = u3[N : N + NM]

# u4 segmentation: [N+2 | NM+2]
u4_pos = u4[0 : N + 2]
u4_neg = u4[N + 2 : N + 2 + NM + 2]

# Compute molar flux jp (positive electrode)
# ------------------------------------------------------------------
# Positive electrode molar flux jp
# ------------------------------------------------------------------
jp = [sympify(0)] * N

for j in range(N):
    theta = u2_pos[j]
    Up = (-0.8090)*theta + 4.4875 \
         - 0.0428  * tanh(18.5138 * (theta - 0.5542)) \
         - 17.7326 * tanh(15.7890 * (theta - 0.3117)) \
         + 17.5842 * tanh(15.9308 * (theta - 0.3120))

    # electrolyte node corresponding to this particle surface is u1_pos[j+1]
    ce = u1_pos[j+1]          # electrolyte concentration at interior node
    phis = u4_pos[j+1]        # solid potential (interior)
    phie = u5_pos[j+1]        # liquid potential

    jp[j] = (
        2 * kp *
        sqrt(ce * c0) *
        sqrt(ctp - theta * ctp) *
        sqrt(theta * ctp) *
        sinh(0.5 * F / (R * T) * (phis - phie - Up))
    )

# ------------------------------------------------------------------
# Negative electrode molar flux jn
# ------------------------------------------------------------------
jn = [sympify(0)] * NM

for j in range(NM):
    theta = u2_neg[j]

    Un = (1.9793 * exp(-39.3631 * theta) +
          0.2482 -
          0.0909  * tanh(29.8538 * (theta - 0.1234)) -
          0.04478 * tanh(14.9159 * (theta - 0.2769)) -
          0.0205  * tanh(30.4444 * (theta - 0.6103)))

    # electrolyte node: +1
    ce = u1_neg[j+1]          # safe: u1_neg length NM+1
    phis = u4_neg[j+1]        # u4_neg length NM+2
    phie = u5_neg[j+1]        # u5_neg length NM+1

    jn[j] = (
        2 * kn *
        sqrt(ce * c0) *
        sqrt(ctn - theta * ctn) *
        sqrt(theta * ctn) *
        sinh(0.5 * F / (R * T) * (phis - phie - Un))
    )

In [6]:
#  Form PDE/ODE equations (electrolyte concentration in positive electrode)
#u1: Electrolyte concentration (mol/m3)

# finite difference spatial derivatives approximations
dudxf1 = 1/(2*h) * (-u1[2] - 3*u1[0] + 4*u1[1])
dudxb1 = 1/(2*h) * (u1[N-1] + 3*u1[N+1] - 4*u1[N])
dudxf1_2 = 1/(2*h2) * (-u1[N+3] - 3*u1[N+1] + 4*u1[N+2])

bc11 = dudxf1
bc21 = D2pos * dudxb1 - D2sep * dudxf1_2

# Initialize eq1 list
eq1 = [None] * len_u1
eq1 = sp.zeros(1, 1 + N + 1 + M + 1 + NM + 1)

eq1[0] = 0 - bc11

for i in range(1, N + 1):
    d2udx21 = (1 / h ** 2) * (u1[i - 1] - 2 * u1[i] + u1[i + 1])
    eq1[i] = diff(u1[i]) - (D2pos * d2udx21 + ap * (1 - t1) * jp[i] / c0) / ep

eq1[N + 1] = 0 - bc21

# Separator
dudxb1_2 = (1 / (2 * h2)) * (u1[N + M] + 3 * u1[N + M + 2] - 4 * u1[N + M + 1])
dudxf1_3 = (1 / (2 * h3)) * (-u1[N + M + 4] - 3 * u1[N + M + 2] + 4 * u1[N + M + 3])
bc31 = D2sep * dudxb1_2 - D2neg * dudxf1_3

for i in range(N + 2, N + M + 2):
    d2udx21 = (1 / h2 ** 2) * (u1[i - 1] - 2 * u1[i] + u1[i + 1])
    eq1[i] = diff(u1[i]) - D2sep * d2udx21 / es

eq1[N + M + 2] = 0 - bc31

# Negative Electrode
dudxb1_3 = (1 / (2 * h3)) * (u1[N + M + NM + 2] + 3 * u1[N + M + NM + 4] - 4 * u1[N + M + NM + 3])
bc41 = dudxb1_3

for i in range(N + M + 3, N + M + NM + 3):
    d2udx21 = (1 / h3 ** 2) * (u1[i - 1] - 2 * u1[i] + u1[i + 1])
    eq1[i] = diff(u1[i]) - (D2neg * d2udx21 + an * (1 - t1) * jn[i] / c0) / en

eq1[N + M + NM + 3] = 0 - bc41

# u2: Surface concentration
eq2 = sp.zeros(1, N + M + NM + 3)

for i in range(1, N + 1):
    eq2[i] = -u2[i] + u3[i] - jp[i] * Rpp / Dsp / 5 / ctp

for i in range(N + M + 3, N + M + NM + 3):
    eq2[i] = -u2[i] + u3[i] - jn[i] * Rpn / Dsn / 5 / ctn

# u3: Average concentration
eq3 = sp.zeros(1, N + M + NM + 3)

for i in range(1, N + 1):
    eq3[i] = diff(u3[i]) + 3 * jp[i] / Rpp / ctp

for i in range(N + M + 3, N + M + NM + 3):
    eq3[i] = diff(u3[i]) + 3 * jn[i] / Rpn / ctn

# u4: Solid potential
eq4 = sp.zeros(1, N + M + NM + 4)

dudxf4 = (1 / (2 * h)) * (-u4[2] - 3 * u4[0] + 4 * u4[1])
dudxb4 = (1 / (2 * h)) * (u4[N - 1] + 3 * u4[N + 1] - 4 * u4[N])

bc14 = dudxf4 + iapp / sigmap
bc24 = dudxb4

eq4[0] = 0 - bc14

for i in range(1, N + 1):
    d2udx24 = (1 / h ** 2) * (u4[i - 1] - 2 * u4[i] + u4[i + 1])
    eq4[i] = d2udx24 - ap * F * jp[i] / sigmap

eq4[N + 1] = 0 - bc24

# Negative
dudxf4_3 = (1 / (2 * h3)) * (-u4[N + M + 4] - 3 * u4[N + M + 2] + 4 * u4[N + M + 3])
dudxb4_3 = (1 / (2 * h3)) * (u4[N + M + NM + 2] + 3 * u4[N + M + NM + 4] - 4 * u4[N + M + NM + 3])

bc34 = dudxf4_3
bc44 = dudxb4_3 + iapp / sigman

eq4[N + M + 2] = 0 - bc34

for i in range(N + M + 3, N + M + NM + 3):
    d2udx24 = (1 / h3 ** 2) * (u4[i - 1] - 2 * u4[i] + u4[i + 1])
    eq4[i] = d2udx24 - an * F * jn[i] / sigman

eq4[N + M + NM + 3] = 0 - bc44

# u5: Liquid phase potential (V)
eq5 = sp.zeros(1, N + M + NM + 4)
# eq5 = {}

dudxf5 = (1 / (2 * h)) * (-u5[2] - 3 * u5[0] + 4 * u5[1])
dudxb5 = (1 / (2 * h)) * (u5[N - 1] + 3 * u5[N + 1] - 4 * u5[N])
dudxf5_2 = (1 / (2 * h2)) * (-u5[N + 1] - 3 * u5[N - 1] + 4 * u5[N])

bc15 = dudxf5
bc25 = Keffp * dudxb5 - Keffs * dudxf5_2

eq5[0] = 0 - bc15
# eq5[0] = Eq(0, bc15)

# for i in range(1, N + 1):
#     dudx1 = (1 / (2 * h)) * (u1[i + 1] - u1[i - 1])
#     dudx4 = (1 / (2 * h)) * (u4[i + 1] - u4[i - 1])
#     dudx5 = (1 / (2 * h)) * (u5[i + 1] - u5[i - 1])
#     eq5[i] = -sigmap * dudx4 - Keffp * dudx5 + (2 * Keffp * R * T * (1 - t1) * dudx1) / (F * u1[i]) - iapp
    
# Internal equations - Positive Electrode
for i in range(2, N + 2):
    dudx1 = (1 / (2 * h)) * (u1[i + 1] - u1[i - 1])
    dudx4 = (1 / (2 * h)) * (u4[i + 1] - u4[i - 1])
    dudx5 = (1 / (2 * h)) * (u5[i + 1] - u5[i - 1])
    eq5[i - 1] = Eq(0, -sigmap * dudx4 - Keffp * dudx5 + (2 * Keffp * R * T * (1 - t1) * dudx1) / (F * u1[i]) - iapp)

# Add boundary equation
eq5[N + 1] = Eq(0, bc25)

# Internal equations - Separator
for i in range(N + 3, N + 2 + M):
    dudx1 = (1 / (2 * h2)) * (u1[i + 1] - u1[i - 1])
    dudx5 = (1 / (2 * h2)) * (u5[i + 1] - u5[i - 1])
    eq5[i - 1] = Eq(0, -Keffs * dudx5 + (2 * Keffs * R * T * (1 - t1) * dudx1) / (F * u1[i]) - iapp)

# Boundary condition - Separator to Negative Electrode
dudxb5_2 = (1 / (2 * h2)) * (u5[N + M + 1] + 3 * u5[N + M + 3] - 4 * u5[N + M + 2])
dudxf5_3 = (1 / (2 * h3)) * (-u5[N + M + 5] - 3 * u5[N + M + 3] + 4 * u5[N + M + 4])
bc35 = Keffs * dudxb5_2 - Keffn * dudxf5_3

eq5[N + M + 3] = Eq(0, bc35)

# Internal equations - Negative Electrode
for i in range(N + M + 4, N + M + NM + 4):
    dudx1 = (1 / (2 * h3)) * (u1[i + 1] - u1[i - 1])
    dudx4 = (1 / (2 * h3)) * (u4[i + 1] - u4[i - 1])
    dudx5 = (1 / (2 * h3)) * (u5[i + 1] - u5[i - 1])
    eq5[i] = Eq(0, -sigman * dudx4 - Keffn * dudx5 + (2 * Keffn * R * T * (1 - t1) * dudx1) / (F * u1[i]) - iapp)

# Final boundary condition
bc45 = u5[N + M + NM + 4]
eq5[N + M + NM + 4] = Eq(0, bc45)


IndexError: list index out of range

In [ ]:
from sympy import symbols, Eq, tanh, diff, rhs, lhs, zeros

# Define symbolic constants
mu = 10**(-3)
q = 1000
t, initime = symbols('t initime')
ff = 1/2 * tanh(q * (t - initime)) + 1/2

# Assumed previously defined variables:
# eq1, eq2, eq3, eq4, eq5 are dicts/lists of Eq() objects

# Create symbolic arrays for modified equations
size_full = 1 + N + 1 + M + 1 + NM + 1
size_part = N + 2 + M + 1 + NM

# eq1
for i in range(size_full):
    if i == 0 or i == N + 1 or i == N + M + 2 or i == size_full - 1:
        eq1[i] = -mu * (diff(rhs(eq1[i]), t) - diff(lhs(eq1[i]), t)) - rhs(eq1[i]) + lhs(eq1[i])
    else:
        eq1[i] = lhs(eq1[i]) - rhs(eq1[i]) * ff

# eq2
for i in range(size_part):
    if 2 <= i <= N + 1 or N + M + 2 <= i < size_part:
        eq2[i] = -mu * (diff(rhs(eq2[i]), t) - diff(lhs(eq2[i]), t)) - rhs(eq2[i]) + lhs(eq2[i])

# eq3
for i in range(size_full - 1):
    if 2 <= i <= N + 1 or N + M + 3 <= i < size_full:
        eq3[i] = lhs(eq3[i]) - rhs(eq3[i]) * ff

# eq4
for i in range(size_full):
    if i in [0, N + 1, N + M + 2, size_full - 1] or 2 <= i <= N + 1 or N + M + 3 <= i < size_full - 1:
        eq4[i] = -mu * (diff(rhs(eq4[i]), t) - diff(lhs(eq4[i]), t)) - rhs(eq4[i]) + lhs(eq4[i])

# eq5
for i in range(size_full):
    if i in [0, N + 1, N + M + 2, size_full - 1] or 2 <= i <= N + 1 or N + 2 <= i <= N + M + 1 or N + M + 3 <= i < size_full - 1:
        eq5[i] = -mu * (diff(rhs(eq5[i]), t) - diff(lhs(eq5[i]), t)) - rhs(eq5[i]) + lhs(eq5[i])


## Optimization Setup

In [ ]:

# Python version of the MATLAB execution, solving, and optimization pipeline for the P2D model

import numpy as np
from sympy import Matrix, symbols, diag, lambdify
from scipy.integrate import solve_ivp
from scipy.optimize import differential_evolution
import matplotlib.pyplot as plt

# Placeholders for your symbolic variables (to be replaced by actual SymPy symbolic equations and variables)
eqn1, eqn2, eqn3, eqn4, eqn5 = [None]*5  # To be replaced with actual symbolic expressions
varsX = symbols('x0:100')  # Placeholder: adjust range and naming according to your model
kk = symbols('k0:8')

# Combine equations
eqs = eqn1 + eqn2[1:N+1] + eqn2[N+2+M+1:] + eqn3[1:N+1] + eqn3[N+2+M+1:] + eqn4[:N+2] + eqn4[1+N+1+M:] + eqn5

# Mass matrix formulation
MM_sym, f_sym = Matrix(eqs).as_explicit()

# Convert symbolic expressions to numerical functions
MM_func = lambdify((varsX, kk), MM_sym, modules='numpy')
f_func = lambdify((varsX, kk), f_sym, modules='numpy')

t = symbols('t')
mu = 1e-3
q = 1000
initime = 200
ff = 0.5 * np.tanh(q*(t - initime)) + 0.5

# Initial guess (adjust sizes according to actual model size)
U = np.zeros(1 + N + 1 + M + 1 + NM + 1 + 1 + N + 1 + M + 1 + NM + 1 + N + NM + N + NM + N + 2 + NM + 2)
U[:1+N+1+M+1+NM+1] = 1
U[1+1+N+1+M+1+NM+1:N+1+N+1+M+1+NM+1] = 0.27
U[1+1+N+1+M+1+NM+1+N:NM+1+N+1+M+1+NM+1+N] = 0.9014
U[1+1+N+1+M+1+NM+1+N+NM:N+1+N+1+M+1+NM+1+N+NM] = 0.27
U[1+1+N+1+M+1+NM+1+N+NM+N:NM+1+N+1+M+1+NM+1+N+NM+N] = 0.9014
U[1+1+N+1+M+1+NM+1+N+NM+N+NM:N+2+1+N+1+M+1+NM+1+N+NM+N+NM] = 4.30430037
U[1+1+N+1+M+1+NM+1+N+NM+N+NM+N+2:NM+2+1+N+1+M+1+NM+1+N+NM+N+NM+N+2] = 0.09202000152
U[1+1+N+1+M+1+NM+1+N+NM+N+NM+N+2+NM+2:] = 0

y0 = U.copy()

# Experimental data
x_exp = np.loadtxt('voltage_exp.txt')

# PSO bounds and parameters
pp = 0.3
init_params = [1, 1.17, 0.18, 215, 4, 3.3, 0.7, 0.7]
lower_bounds = [(1-pp)*p for p in init_params]
upper_bounds = [(1+pp)*p for p in init_params]

# Objective function
def P2Dobj(kk_):
    try:
        M0 = MM_func(y0, kk_)
        vw = 1 / np.maximum(np.abs(M0).max(axis=1), 1e-10)
        mw = np.diag(vw)

        def F(t, y):
            return vw * f_func(y, kk_)

        def M1(t, y):
            return mw @ MM_func(y, kk_)

        t_span = (0, Totexp + 200)
        t_eval = np.linspace(*t_span, Numexp + 3)

        sol = solve_ivp(F, t_span, y0, method='BDF', t_eval=t_eval, atol=1e-5, rtol=1e-5)
        V_model = sol.y[var1_index] - sol.y[var2_index]  # Fill with correct indices
        return np.sqrt(np.mean((x_exp[:, 0] - V_model[3:])**2))
    except:
        return 1000

# Run PSO (replaced with differential evolution for Python)
result = differential_evolution(P2Dobj, bounds=list(zip(lower_bounds, upper_bounds)), strategy='best1bin',
                                maxiter=10, popsize=10, disp=True)
kk_opt = result.x

# Extracted parameters
D1, Kappa, sigmap, sigman, Dsp, Dsn, kp, kn = kk_opt
D1 *= 1e-9
Dsp *= 1e-15
Dsn *= 1e-14
kp *= 1e-11
kn *= 1e-12



## Simulate the Model

In [ ]:
# Final solve using optimal params
M0 = MM_func(y0, kk_opt)
vw = 1 / np.maximum(np.abs(M0).max(axis=1), 1e-10)
mw = np.diag(vw)

F = lambda t, y: vw * f_func(y, kk_opt)
M1 = lambda t, y: mw @ MM_func(y, kk_opt)

sol = solve_ivp(F, (0, 100000), y0, method='BDF', atol=1e-5, rtol=1e-5)

## Plotting and Validation

# Plot
plt.figure(figsize=(10, 6))
plt.plot(sol.t - initime, sol.y[var1_index] - sol.y[var2_index], label='P2D Model', linewidth=2)
plt.plot(np.linspace(0, Totexp, Numexp), x_exp[:, 0], 'ro', label='Experiment')
plt.xlabel('Time (s)')
plt.ylabel('Voltage (V)')
plt.legend()
plt.grid(True)
plt.savefig('voltage_25C.bmp')
plt.show()